# AI² at Gravitas — Build a Financial Research Agent

You are joining a small research desk. Your brief:

> **Build an agent that can answer evidence-backed questions about TCS, Infosys and HCLTech — and show where every important claim came from.**

This notebook is the **main workshop surface**. The slides introduce ideas; here you will immediately use them.

### The rhythm for today

**Read → predict → fill a small stub → run → inspect → explain what changed.**

You are *not* expected to write an entire AI system from a blank file. The starter code handles plumbing so you can focus on the important design choices.


## How to use this notebook
Throughout the notebook you will see five kinds of blocks:
- **🧠 Concept** — the minimum idea you need before touching the code.
- **🔮 Predict** — make a guess before you run something.
- **✍️ Your turn** — a deliberately small coding/design task, usually 2–8 minutes.
- **🔎 Inspect** — examine evidence, rankings, intermediate outputs and traces, not only “it ran.”
- **✅ Checkpoint** — explain the idea back in your own words before moving on.

### Your build map
You will make many small decisions and implement three focused Python files rather than merely running completed cells:
1. classify knowledge-cutoff vs hallucination failures;
2. design a cache-friendly prompt layout;
3. build a structured `ResearchAnswer`;
4. choose fine-tuning vs RAG for different requirements;
5. format provenance into a citation label;
6. inspect parsing across several pages;
7. measure two chunking configurations;
8. implement `student_work/indexing.py`;
9. reason about HNSW vs IVFFlat trade-offs;
10. use metadata filtering in vector search;
11. close and score a real RAG loop;
12. implement RRF in `student_work/retrieval.py`;
13. diagnose dense vs keyword vs reranked results;
14. implement deterministic guidance arithmetic;
15. assemble the Agno agent in `student_work/agent.py`;
16. predict and inspect its tool path;
17. design a subagent delegation plan;
18. test a guardrail;
19. classify tool vs skill vs sandbox;
20. design a context-compaction policy;
21. diagnose a Langfuse trace;
22. improve one measured weakness in the capstone.

<details><summary><b>If you get stuck</b></summary>
Ask OpenCode to explain the surrounding code, state what you expect, or give one hint. Do not ask it to replace the whole exercise. Read the diff, run the tests, and explain the change back in your own words.
</details>


In [ ]:
from pathlib import Path
import sys

HERE = Path.cwd().resolve()
candidates = [
    HERE,
    HERE.parent,
    HERE.parent.parent,
    HERE.parent / 'Gravitas_Workshop_Starter',
    HERE.parent.parent / 'Gravitas_Workshop_Starter',
]
ROOT = next((p for p in candidates if (p / 'data/corpus_manifest.json').exists()), None)
if ROOT is None:
    raise FileNotFoundError('Could not find Gravitas_Workshop_Starter. Keep the notebook in the starter kit, or extract the instructor pack beside the starter folder.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print('Workshop root:', ROOT)
print('Notebook ready:', (ROOT / 'data/corpus_manifest.json').exists())

from observability.langfuse import flush_langfuse, dashboard_base_url


# Part I — Start with the simplest possible AI system

## Mission 1 — What does a plain LLM actually know?

### 🧠 Concept: an LLM generates text; it is not your evidence database

A language model is very good at producing useful text from its learned parameters and the context you send in the current request. But for financial research, we care about a second question:

> **Where did this claim come from?**

A fluent answer can still be stale, approximate, or unsupported.

<img src="../assets/notebook/llm_next_token.png" width="900" alt="Workshop slide explaining next-token language modelling">


### 🧠 Two different failure modes: knowledge cutoff vs hallucination
**Knowledge cutoff / stale knowledge** means model weights do not automatically absorb a filing released after that model version was trained or updated.  
**Hallucination** means the model generates a claim that is unsupported or wrong. It can hallucinate about old facts or new facts, and even after retrieval it can misread evidence.

```text
knowledge cutoff → the model may not have the fact
hallucination    → the model may say a fact it cannot support
```
RAG helps with freshness and can reduce hallucination, but does not eliminate it.


In [ ]:
failure_types = {
    'A filing was published after the model version was trained': None,  # TODO
    'The model invents a page number for a real annual report': None,  # TODO
    'The correct passage is retrieved, but the model changes 12.4% to 14.2%': None,  # TODO
}
failure_types


### 🔮 Predict before running

For the question below, write down one thing you think the model might get wrong or fail to prove.

> **Question:** What was TCS FY25 revenue and how fast did it grow in constant currency?

Possible failure categories: **freshness / exact number / unit / period / source / citation / unsupported confidence**.


In [ ]:
# ✍️ YOUR TURN 1 — You may change this to another filing-style question.
question = 'What was TCS FY25 revenue and how fast did it grow in constant currency?'
question


In [ ]:
from llm.client import call_model
from llm.structured import build_messages

plain_answer = call_model(build_messages(question))
print(plain_answer)


In [ ]:
# 🔭 TRACE CHECKPOINT 1 — make sure the generation reached Langfuse.
flush_langfuse()
print("Langfuse:", dashboard_base_url())
print("Open Tracing → Traces and find the latest workshop-llm-call.")


### 🔭 TRACE CHECKPOINT 1 — plain LLM

Open Langfuse **now**, while the system is still simple.

Find the latest trace/generation and inspect:

1. What input did the model receive?
2. How long did the generation take?
3. Is there any retrieval span? **There should not be one yet.**
4. Can the trace tell you where the factual answer came from?

Keep the dashboard tab open. We will return to the same execution view after adding RAG and agents.

> **Trace** = one end-to-end execution.  
> **Span / observation** = one meaningful step inside that execution, such as retrieval, reranking, a tool call, or a generation.


### 🔎 Inspect the answer — not just whether it sounds good

Ask yourself:

- Does it tell you **which document** supports the number?
- Can you inspect the **page or source**?
- If it gives a precise percentage, do you know whether it came from evidence or model memory?
- If the answer is wrong, can you tell *why* it was wrong?

**Key idea:** today we will make the evidence path increasingly inspectable.


## Concept checkpoint — Prompt caching: reuse repeated prefixes, not answers
Long requests often repeat the same **system instructions, tool schemas, and examples**. Many providers can reuse computation for an identical/stable prompt prefix.

```text
stable prefix: system instructions + tool schemas + reusable examples
dynamic suffix: current question + fresh retrieved evidence
```

Prompt caching can reduce repeated input work, latency, or cost depending on the provider. It does **not** update model knowledge, extend context, or guarantee correctness. It is different from caching the final answer.


In [ ]:
cache_plan = {
    'stable_prefix': [],   # TODO: add 2-3 repeated items
    'dynamic_suffix': [],  # TODO: add changing items
}
cache_plan


## Mission 2 — Structured outputs: turn prose into a software interface

### 🧠 Concept

Humans can interpret a paragraph. Software needs predictable fields.

For our research system we want an answer shaped like:

```text
ResearchAnswer
├─ answer
├─ citations[]
│  ├─ source_id
│  └─ page
└─ confidence
```

Structure improves **reliability of the interface**. It does **not** make the facts true by itself.


In [ ]:
from llm.schemas import ResearchAnswer, Citation

ResearchAnswer.model_json_schema()


In [ ]:
# ✍️ YOUR TURN 2 — Build one valid object manually.
# Hint: confidence must be 'low', 'medium' or 'high'.
structured_example = None  # TODO: create a ResearchAnswer(...)
structured_example


<details><summary><b>Hint</b></summary>
Create a `Citation(...)` first or inline it inside `ResearchAnswer(...)`. Try `source_id='example-source'`, `page=1` and `confidence='low'`.
</details>

✅ **Checkpoint:** Why can a perfectly valid JSON/structured answer still be hallucinated?


## Concept checkpoint — Fine-tuning vs RAG: change behaviour or supply evidence?
**Fine-tuning changes model weights.** It is useful when examples define repeated behavior/style/task patterns.  
**RAG keeps the model fixed and supplies external evidence at inference time.** It is useful when knowledge changes, belongs to private documents, or needs citations/provenance.

| Question | Fine-tuning | RAG |
|---|---|---|
| Shape repeated behaviour/style? | Strong fit | Limited |
| Add tomorrow's filing without retraining? | Poor fit | Strong fit |
| Inherent source citations? | No | It can |
| Main burden | data + training + eval | ingestion + retrieval + eval |

They can be combined.


In [ ]:
adaptation_choices = {
    'Answer from a quarterly filing published tomorrow with page citations': None,  # TODO
    'Always classify support tickets into a fixed company taxonomy': None,  # TODO
    'Use private filings AND follow a specialized answer style': None,  # TODO
}
adaptation_choices


# Part II — Build the evidence layer before building RAG

## Mission 3 — Define the corpus

### 🧠 Concept: RAG starts with a *library*, not with a vector database

A **corpus** is the set of documents your system is allowed to search. Ours contains official investor documents for TCS, Infosys and HCLTech.

Each piece of evidence must preserve **provenance**: company, period, document type, filename, page and chunk ID. Otherwise a good-looking answer becomes impossible to audit.


In [ ]:
import json
manifest = json.loads((ROOT / 'data/corpus_manifest.json').read_text())
[(x['company'], x['doc_type'], x['period'], x['filename']) for x in manifest]


### ✍️ Your turn — turn metadata into a citation label
Provenance is only useful if the application can carry it all the way to the user. Write a tiny formatter before we ever touch embeddings.


In [ ]:
def citation_label(record):
    # TODO: include company, doc_type, period, page, filename
    return None
demo_record = {**manifest[0], 'page': 3}
citation_label(demo_record)


### Acquire the official PDFs

The notebook orchestrates the workflow, but downloading belongs in a terminal.

```bash
uv run python scripts/fetch_corpus.py
```

If venue internet is slow, your instructor may give you the pre-downloaded `data/documents/` checkpoint. That is a recovery path, not a different lesson.


In [ ]:
docs = sorted((ROOT / 'data/documents').glob('*.pdf'))
[(p.name, round(p.stat().st_size / 1_000_000, 2)) for p in docs]


## Mission 4 — Parsing: turn page layouts into evidence text

### 🧠 Concept

A PDF is a **visual page format**, not a clean text database. Financial filings may contain columns, tables, repeated headers, scanned pages and page-number mismatches.

Parsing is the stage where we ask:

> Did the evidence survive conversion into machine-readable text?

We will first test a short Infosys release before processing everything.


In [ ]:
from retrieval.parsing import parse_pdf_pages

sample = ROOT / 'data/documents/Infosys_Q4_FY25_Press_Release.pdf'
pages = parse_pdf_pages(sample)
print('Parsed pages:', len(pages))


In [ ]:
# ✍️ YOUR TURN 3 — Inspect the parser instead of trusting it blindly.
page_to_inspect = 0      # TODO: try another page after your first run
search_term = 'revenue'  # TODO: try 'margin', 'guidance' or another finance term

text = pages[page_to_inspect]['text']
print(text[:2200])
print('\nContains search term?', search_term.lower() in text.lower())


### 🔎 What to inspect

Look for one example of each:

- a number that survived correctly,
- a heading or sentence boundary that survived,
- a layout/table artifact that became messy.

If parsing destroys the evidence, no retrieval algorithm later can magically recover it.


### ✍️ Your turn — inspect more than the first page
A parser can succeed on page 1 and silently fail later. Write a tiny helper that finds pages containing a finance term, then inspect one of those pages manually.


In [ ]:
def pages_with_term(parsed_pages, term):
    # TODO: return page numbers containing term case-insensitively
    return []
pages_with_term(pages, 'margin')[:10]


## Mission 5 — Chunking: decide what becomes retrievable

### 🧠 Concept

We cannot send hundreds of pages to the model for every question. So we split parsed pages into smaller evidence units called **chunks**.

- **Too small:** a number may be separated from the sentence that explains it.
- **Too large:** retrieval gets noisy and context becomes expensive.
- **Overlap:** repeats a little text across boundaries so important sentences are less likely to be cut in half.

<img src="../assets/notebook/chunking.png" width="900" alt="Workshop slide illustrating chunk size and overlap">


In [ ]:
from retrieval.chunking import chunk_pages

# ✍️ YOUR TURN 4 — These are design choices, not magic constants.
CHUNK_SIZE = 1200  # TODO: after the first run, try 700 or 1700
OVERLAP = 120      # TODO: after the first run, try 0 or ~10% of chunk size

chunks = chunk_pages(pages, chunk_size=CHUNK_SIZE, overlap=OVERLAP)
print('Chunks:', len(chunks))
[(c['page'], c['chunk'], c['text'][:220].replace('\n',' ')) for c in chunks[:5]]


### 🔮 Mini experiment

Change **one parameter at a time** and rerun the previous cell.

Predict first:

- If chunk size gets smaller, will the number of chunks go **up or down**?
- If overlap gets larger, will repeated text go **up or down**?
- Which setting looks easier to cite and understand?

There is no universally correct chunk size; the right answer depends on the documents and questions.


### ✍️ Your turn — measure the chunking trade-off
Do not pick a chunk size only because a tutorial used it. Compare two settings using observable statistics.


In [ ]:
def chunk_stats(rows):
    # TODO: return count, avg, min and max char length
    return {}
chunks_small = chunk_pages(pages, chunk_size=700, overlap=70)
chunks_large = chunk_pages(pages, chunk_size=1700, overlap=170)
{'small': chunk_stats(chunks_small), 'large': chunk_stats(chunks_large)}


## Mission 6 — Build the processed corpus

Now apply the same parsing + chunking pipeline to all eight official PDFs and attach metadata.

This is the point where “a folder of PDFs” becomes **searchable evidence objects**.


In [ ]:
from retrieval.corpus import build_corpus, load_corpus

# Full-corpus parsing can be the slowest local step.
# If the room stalls, use the facilitator checkpoint in RECOVERY.md and run load_corpus().
corpus = build_corpus(chunk_size=CHUNK_SIZE, overlap=OVERLAP)
print('Corpus chunks:', len(corpus))
print('Example metadata:', {k: corpus[0][k] for k in ['source_id','company','period','doc_type','page','chunk']})


✅ **Checkpoint:** If a retrieved sentence says “revenue grew 4.2%”, what metadata would you need to show a defensible citation to a user?


## Mission 7 — Indexing and vector search

### 🧠 Concept: semantic search compares meaning

An embedding model maps text into numerical representations. Similar meanings tend to land near one another in that representation space.

We use Pinecone's integrated embedding path so this workshop can focus on retrieval design instead of embedding-service plumbing.

After the corpus exists, first complete the indexing-record exercise below. Then open a terminal:

```bash
uv run python scripts/index_corpus.py
```


### ✍️ YOUR TURN — build the record that will enter the vector index
Before running the indexing script, open `student_work/indexing.py` and implement `make_pinecone_record(row)`.
Preserve `_id = source_id`, text, source_id, company, period, document type, filename and page.


In [ ]:
!uv run pytest -q tests/test_student_indexing.py


### Why metadata still matters

Vector similarity answers **“what sounds semantically relevant?”** Metadata answers **“which company / period / document type is even eligible?”**

Good retrieval systems usually use both.


### 🧠 Brief detour — HNSW and IVFFlat
Vector stores usually avoid comparing a query against every vector. **Approximate nearest-neighbour (ANN) indexes** trade a little recall for much faster search.
- **HNSW** builds a multi-layer graph. It usually offers a strong query speed/recall trade-off, but costs more memory and takes longer to build.
- **IVFFlat** partitions vectors into lists/clusters and probes only some of them. It builds faster and uses less memory, but recall depends more on list/probe tuning.
You are **not implementing either index today**. The goal is simply to recognize that vector search itself has an index and a speed/recall trade-off.


In [ ]:
ann_choices = {
    'Prioritize query quality/speed and can afford more memory/build time': None,  # TODO
    'Prioritize fast builds/lower memory and can tune probes for recall': None,  # TODO
}
ann_choices


## Mission 8 — RAG: retrieve first, then generate

### 🧠 Concept

RAG is not “training the model on your PDFs.” The documents stay outside the model. At question time we:

1. search for relevant chunks,
2. place selected evidence into the model context,
3. ask the model to answer using that evidence.

<img src="../assets/notebook/rag_pipeline.png" width="900" alt="Workshop slide showing the RAG pipeline">


## Concept checkpoint — Why RAG became such a common LLM application pattern
Retrieval + generation has older roots, but the **RAG name and influential formulation arrived in 2020**. It combined a generator with an external retrievable memory and highlighted two limitations of parameter-only knowledge: updating knowledge and showing provenance.
As LLM applications spread, teams wanted private/fresh data without retraining and answers tied to inspectable evidence. Embedding services and vector stores made the pattern practical to ship.

**Advantages:** fresh/private evidence, provenance/citations, modular updates, question-specific context.  
**Disadvantages:** more moving parts; parsing/chunking/retrieval can fail; extra latency/cost; retrieved text can be noisy or malicious; generation can still hallucinate.


In [ ]:
rag_failure_owner = {
    'A table row disappears during PDF extraction': None,  # TODO
    'The correct chunk exists but is not in top-k': None,  # TODO
    'Correct evidence is supplied but the answer invents another number': None,  # TODO
}
rag_failure_owner


In [ ]:
from retrieval.search import pinecone_search, keyword_search

# ✍️ YOUR TURN 5 — Retrieval has knobs too.
query = 'What FY26 revenue growth guidance did Infosys give?'  # TODO: change later
TOP_K = 5  # TODO: try 3 vs 8 and inspect noise

dense = pinecone_search(query, top_k=TOP_K)
[(x['company'], x['doc_type'], x['page'], x['text'][:180].replace('\n',' ')) for x in dense]


### 🔎 Inspect *before* generating

Do not jump straight to the final answer. Check the retrieved evidence first:

- Is the correct company present?
- Is the correct period present?
- Did the relevant guidance sentence make the top results?
- Are there near-duplicate chunks wasting context?

If retrieval is bad, generation cannot repair missing evidence.


### ✍️ Your turn — metadata filter before similarity search
If the question explicitly says **Infosys FY26**, there is little value in letting TCS FY25 chunks compete for the first-stage shortlist.
Fill the company/period filter and compare the result list with the unfiltered search above.


In [ ]:
metadata_filter = {
    'company': {'$eq': 'TODO'},  # TODO
    'period': {'$eq': 'TODO'},   # TODO
}
filtered_dense = pinecone_search(query, top_k=TOP_K, filter=metadata_filter)
[(x['company'], x['period'], x['page'], x['text'][:130].replace('\n',' ')) for x in filtered_dense]


### 🔎 Compare the two result lists
Did metadata filtering remove irrelevant companies/periods? Did it accidentally exclude a useful cross-company result?
**Important:** filters improve precision only when the metadata and user intent are both correct. They can also hide evidence if applied too aggressively.


In [ ]:
from retrieval.rag import build_context

context = build_context(dense)
print(context[:3200])


In [ ]:
from langfuse import observe

# ✍️ YOUR TURN 6 — Close the basic dense-RAG loop.
# Hybrid retrieval comes next; for now keep the path simple: dense search → context → generation.
@observe(name="rag-checkpoint")
def run_rag(question):
    trace_hits = pinecone_search(question, top_k=4)
    trace_context = build_context(trace_hits)
    return None  # TODO: call the same model with build_messages(question, context=trace_context)

grounded_answer = run_rag(query)
print(grounded_answer)
flush_langfuse()


In [ ]:
# ✍️ YOUR TURN 7 — Score the *system*, not the writing style.
rag_check = {
    'retrieved_relevant_evidence': None,      # TODO: True/False
    'answer_uses_supplied_evidence': None,    # TODO: True/False
    'source_ids_are_visible_or_traceable': None,
    'more_inspectable_than_plain_llm': None,
}
rag_check


### 🔭 TRACE CHECKPOINT 2 — RAG

Refresh Langfuse and open the latest `rag-checkpoint` trace.

This time you should see a **small execution tree**:

```text
rag-checkpoint
├─ pinecone-search
└─ workshop-llm-call / generation
```

Answer with a partner:

- Which span selected evidence?
- Which step took longer?
- What changed compared with the plain-LLM trace?
- If the answer were wrong, would you inspect retrieval or generation first—and why?

We intentionally keep this first RAG trace simple. Next we will add keyword retrieval, fusion, and reranking and watch the trace become richer.


✅ **Checkpoint:** RAG mainly fixes the problem **“the model did not have the evidence.”**

It does **not** automatically fix:

- bad parsing,
- bad retrieval,
- wrong document version,
- incorrect reasoning over correct evidence,
- unsupported claims the model adds anyway.

That is why we debug RAG as a pipeline.


## Mission 9 — Hybrid retrieval: finance needs meaning *and* exact strings

### 🧠 Concept

Semantic/vector search is excellent for questions such as **“why did margins weaken?”**

Finance also contains exact tokens that matter disproportionately: **24.3%**, **₹**, quarter names, tickers, guidance ranges and accounting terms.

Hybrid retrieval combines:

- **dense/vector search** for meaning,
- **keyword/BM25 search** for exact lexical matches,
- **fusion** to combine ranked lists,
- **reranking** to spend more compute on a small shortlist.

<img src="../assets/notebook/hybrid_retrieval.png" width="900" alt="Workshop slide showing hybrid dense and keyword retrieval">


### Reciprocal Rank Fusion (RRF)

Dense and keyword systems produce scores on different scales, so comparing raw scores directly can be awkward. RRF instead rewards items that appear near the top of one or more ranked lists.

For each result at rank `r`:

```text
contribution = 1 / (k + r)
```

Then add contributions across rankings.

### ✍️ YOUR TURN 8 — implement RRF

Open:

```text
student_work/retrieval.py
```

Replace the `pass` in `reciprocal_rank_fusion(...)`.

Use this mental algorithm:

```text
scores = {}
for each ranking:
    for each source_id with rank starting at 1:
        add 1 / (k + rank)
sort highest score first
```

Then run the test below.


In [ ]:
!uv run pytest -q tests/test_student_retrieval.py


In [ ]:
from student_work.retrieval import reciprocal_rank_fusion

reciprocal_rank_fusion([['a','b','c'], ['b','d','a']])


### Apply the same idea to real evidence

We will now compare the stages rather than treating “search” as one black box.

> **Teaching note:** `retrieval/ranking.py` uses a deliberately transparent lexical-overlap scorer so you can inspect the ranking stage. Production systems often use a learned reranker.


In [ ]:
from retrieval.ranking import rerank_candidates

@observe(name="hybrid-retrieval-checkpoint")
def run_hybrid_retrieval(question):
    dense_hits = pinecone_search(question, top_k=8)
    keyword_hits = keyword_search(question, top_k=8)

    by_id = {h['source_id']: h for h in dense_hits + keyword_hits}
    fused = reciprocal_rank_fusion([
        [h['source_id'] for h in dense_hits],
        [h['source_id'] for h in keyword_hits],
    ])
    fused_hits = [by_id[sid] for sid, _score in fused if sid in by_id]
    final_hits = rerank_candidates(question, fused_hits, top_k=5)
    return dense_hits, keyword_hits, final_hits

dense, keyword, reranked = run_hybrid_retrieval(query)
flush_langfuse()

print('Dense top 3:')
for h in dense[:3]:
    print('-', h['source_id'], h['text'][:120].replace('\n', ' '))

print('\nKeyword top 3:')
for h in keyword[:3]:
    print('-', h['source_id'], h['text'][:120].replace('\n', ' '))

print('\nAfter fusion + reranking:')
for h in reranked:
    print('-', h['source_id'], h['text'][:150].replace('\n', ' '))


### 🔭 TRACE CHECKPOINT 2B — hybrid retrieval

Open `hybrid-retrieval-checkpoint` in Langfuse.

Now the trace should make the retrieval architecture visible:

```text
hybrid-retrieval-checkpoint
├─ pinecone-search
├─ keyword-search
└─ rerank-candidates
```

This is a useful debugging view because **“retrieval failed” is no longer one thing**. Dense search, keyword search, fusion/reranking, or metadata can each be the weak link.

Find one query where the dense and keyword branches disagree. Which branch contributed the evidence you actually wanted?


In [ ]:
# ✍️ YOUR TURN 9 — Diagnose the retrieval change in your own words.
retrieval_observation = 'TODO: write 1–2 sentences about what changed between dense, keyword and reranked results.'
print(retrieval_observation)


# Part III — Give the model actions, not just evidence

## Mission 10 — Tools: deterministic work should not be a language-model guess

### 🧠 Concept

A tool is an ordinary function the model is allowed to request.

If we need arithmetic, retrieval or market data, it is better to expose a bounded capability than ask the model to “do it in its head.”

For this workshop we use:

- `CalculatorTools`
- `search_documents()`
- `read_source()`
- `market_snapshot()`


### ✍️ Your turn — make deterministic arithmetic explicit
Before giving arithmetic to an agent, write the bounded operation as ordinary code. This is the mental model behind tools: **the model chooses the operation; deterministic code performs it.**


In [ ]:
def guidance_midpoint(low, high):
    # TODO: return midpoint
    return None
def midpoint_gap(range_a, range_b):
    # TODO: midpoint(A) - midpoint(B)
    return None
print('Midpoint:', guidance_midpoint(1.0, 3.0))
print('Gap:', midpoint_gap((1.0, 3.0), (2.0, 4.0)))


In [ ]:
from agno.tools.calculator import CalculatorTools
from tools.research import search_documents
from retrieval.corpus import read_source
from tools.market import market_snapshot

calculator = CalculatorTools(include_tools=['add','subtract','multiply','divide'])
print('Calculator toolkit:', calculator.name)
print('Research tools:', search_documents.__name__, read_source.__name__, market_snapshot.__name__)


### 🔮 Predict

Question: **“Compare Infosys and HCLTech FY26 guidance and calculate the midpoint gap.”**

A good system may need more than retrieval:

1. search evidence for company A,
2. search evidence for company B,
3. read the relevant source(s),
4. calculate a comparison,
5. write an evidence-backed answer.

A fixed one-shot RAG call does not naturally decide and repeat these actions. That leads us to an agent loop.
